# 03 — Perception review

Visually review deterministic media facts, shot boundaries, and representative frames.

Model-free: ffprobe + PySceneDetect + ffmpeg only. See `docs/perception.md`.

In [ ]:
# Bootstrap: make the local package importable when running from the repo.
import sys
from pathlib import Path

_src = Path.cwd()
if _src.name == 'notebooks':
    _src = _src.parent
if (_src / 'src' / 'tiktok_analytics_factory').is_dir():
    sys.path.insert(0, str(_src / 'src'))
import tiktok_analytics_factory  # noqa: E402
print(tiktok_analytics_factory.__file__)

In [ ]:
from pathlib import Path

# Canonical reference video ingested for issue #2.
VIDEO_ID = "6718335390845095173"
REFERENCE_VIDEO = Path("data/raw") / VIDEO_ID / "video.mp4"
if not REFERENCE_VIDEO.is_file():
    raise FileNotFoundError(
        f"Canonical reference video missing: {REFERENCE_VIDEO}. "
        "Run the issue #2 ingestion first; no synthetic substitute is used here."
    )
print(f"using canonical reference video: {REFERENCE_VIDEO}")


## 1. Run the perception pipeline

In [ ]:
from tiktok_analytics_factory.perception import run_perception

OUTPUT_DIR = Path("data/derived") / REFERENCE_VIDEO.stem / "perception" / "perception_v1"
manifest = run_perception(REFERENCE_VIDEO, OUTPUT_DIR)
manifest.to_dict().keys()

## 2. Media facts

In [ ]:
import json
print(json.dumps(manifest.media_facts.to_dict(), indent=2))

## 3. Shot boundaries

In [ ]:
for s in manifest.shots.shots:
    print(f"{s.shot_id}: {s.start_seconds:8.3f}s -> {s.end_seconds:8.3f}s "
          f"(frames {s.start_frame}-{s.end_frame})")

## 4. Manual hard-cut annotation

Edit this list after watching the video once: timestamps (seconds) of **hard visual cuts** only.

In [ ]:
# Hand-annotated hard visual cuts for video 6718335390845095173.
# Produced by reviewing before/after frames at each boundary and a 4 fps contact sheet.
MANUAL_CUTS = [0.50, 1.13, 3.40, 4.20, 4.93, 5.70, 6.60, 7.57, 8.37]
# Persisted alongside the perception artifacts:
MANUAL_CUTS_SOURCE = Path("data/derived") / VIDEO_ID / "perception" / "v1" / "hard_cut_annotation.json"
MANUAL_CUTS


## 5. Boundary quality vs manual annotation (tolerance ±0.30 s)

In [ ]:
from tiktok_analytics_factory.perception import evaluate_boundaries
from tiktok_analytics_factory.perception.evaluation import CutAnnotation, shot_cut_timestamps

detected = shot_cut_timestamps([(s.start_seconds, s.end_seconds) for s in manifest.shots.shots])
ev = evaluate_boundaries(detected, [CutAnnotation(t) for t in MANUAL_CUTS],
                         tolerance_seconds=0.30)
ev.to_dict()

## 6. Visual review of representative frames

In [ ]:
from IPython.display import display, Image

for art in manifest.frames:
    print(f"{art.shot_id} @ {art.timestamp_seconds:.3f}s")
    display(Image(art.path, width=240))